# Proyecto 4: Clasificación de Sentimientos con NLP

## 🎯 Objetivos
- Implementar un pipeline completo de Procesamiento de Lenguaje Natural (NLP).
- Resolver el problema del desbalance de clases mediante *Undersampling*.
- Comparar diferentes métodos de vectorización de texto: *Bag of Words* vs *TF-IDF*.
- Evaluar y comparar múltiples modelos de Machine Learning (SVM, Decision Tree, Naive Bayes, Logistic Regression).
- Optimizar hiperparámetros utilizando `GridSearchCV`.

## 1. Introducción

El análisis de sentimientos es una tarea fundamental de NLP que consiste en determinar la polaridad de un texto (positivo, negativo o neutro). 

El reto principal es que las computadoras no entienden palabras, sino números. Por lo tanto, el proceso consiste en transformar texto no estructurado en una matriz numérica que el modelo pueda procesar, un proceso conocido como **Vectorización**.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.under_sampling import RandomUnderSampler

DATA_PATH = Path('IMDB Dataset.csv')
df_review = pd.read_csv(DATA_PATH)
df_review.head()

## 2. Gestión del Desbalance de Clases

Un modelo entrenado con un dataset desbalanceado tenderá a predecir siempre la clase mayoritaria, ignorando la minoritaria.

In [ ]:
# Analizar la distribución actual
print("Distribución original:\n", df_review['sentiment'].value_counts())

# Crear un dataset desbalanceado para demostración (9000 pos / 1000 neg)
df_pos = df_review[df_review['sentiment'] == 'positive'][:9000]
df_neg = df_review[df_review['sentiment'] == 'negative'][:1000]
df_desbalanced = pd.concat([df_pos, df_neg])

# Aplicar RandomUnderSampler para balancear
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(df_desbalanced[['review']], df_desbalanced['sentiment'])

df_balanced = pd.DataFrame({'review': X_resampled['review'], 'sentiment': y_resampled})
print("\nDistribución balanceada:\n", df_balanced['sentiment'].value_counts())

## 3. Vectorización de Texto

### El Puente Pedagógico: BoW vs TF-IDF

Para convertir texto en números, existen dos enfoques comunes:

1. **Bag of Words (CountVectorizer)**: Simplemente cuenta cuántas veces aparece cada palabra. El problema es que palabras comunes como "el", "de", "que" (stop words) tendrán los conteos más altos pero no aportan significado.
2. **TF-IDF (Term Frequency - Inverse Document Frequency)**: Penaliza las palabras que aparecen en casi todos los documentos y resalta aquellas que son únicas y descriptivas de un documento específico.

#### Flujo de Procesamiento de NLP
```
  [ Raw Text ] --> [ Preprocessing ] --> [ Vectorization ] --> [ ML Model ] --> [ Prediction ]
                          |                    |                    |
                    (Cleaning, Stopwords) (TF-IDF / BoW)       (SVM, LR, etc.)
```

In [ ]:
# División de datos
train, test = train_test_split(df_balanced, test_size=0.33, random_state=42)
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

# Usamos TF-IDF para una mejor representación
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

## 4. Comparativa de Modelos

Probaremos cuatro algoritmos clásicos para ver cuál se adapta mejor a la naturaleza del texto.

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

models = {
    "SVM": SVC(kernel='linear'),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression()
}

scores = {}
for name, model in models.items():
    # Naive Bayes requiere array denso
    X_train = train_x_vector.toarray() if name == "Naive Bayes" else train_x_vector
    X_test = test_x_vector.toarray() if name == "Naive Bayes" else test_x_vector
    
    model.fit(X_train, train_y)
    scores[name] = model.score(X_test, test_y)

print("Resultados de Accuracy:\n", scores)

## 5. Evaluación Detallada (SVM)

El Accuracy puede ser engañoso. Utilizamos la **Matriz de Confusión** y el **F1-Score** para entender dónde falla el modelo (Falsos Positivos vs Falsos Negativos).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = models["SVM"]
predictions = best_model.predict(test_x_vector)

print("Reporte de Clasificación:\n", classification_report(test_y, predictions))
print("Matriz de Confusión:\n", confusion_matrix(test_y, predictions))

## 6. Optimización con GridSearchCV

Los modelos tienen "perillas" llamadas hiperparámetros. `GridSearchCV` prueba automáticamente todas las combinaciones posibles para encontrar la mejor configuración.

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'C': [1, 4, 8, 16, 32], 'kernel': ['linear', 'rbf']}
grid = GridSearchCV(SVC(), params, cv=5)
grid.fit(train_x_vector, train_y)

print(f"Mejores parámetros: {grid.best_params_}")
print(f"Mejor score: {grid.best_score_:.4f}")

## 📝 Ejercicios

1. **Test Personalizado**: Crea una lista de 3 frases (algunas positivas y otras negativas) y usa `tfidf.transform()` y `svc.predict()` para ver si el modelo acierta.
2. **Impacto de Stopwords**: Entrena el modelo nuevamente pero esta vez SIN `stop_words='english'`. ¿Sube o baja la precisión?
3. **Ajuste de Hiperparámetros**: Añade el parámetro `gamma` al `GridSearchCV` para el kernel `rbf` y observa si el score mejora.

## 📋 Resumen de Modelos de Clasificación de Texto

| Modelo | Fortalezas | Debilidades | Recomendación |
|---|---|---|---|
| **SVM** | Muy robusto en alta dimensionalidad | Lento en datasets gigantes | Texto corto/medio |
| **Decision Tree** | Fácil de interpretar | Tiende al Overfitting | Datos tabulares |
| **Naive Bayes** | Extremadamente rápido | Asume independencia de palabras | Baselines rápidos |
| **Logistic Reg.** | Probabilístico y eficiente | Simple para relaciones complejas | Clasificación binaria |